In [1]:
import yaml
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# Load database config from YAML
with open("../configs/database.yaml", "r") as f:
    config = yaml.safe_load(f)

pg_conf = config["database"]["postgres"]
pg_user = quote_plus(pg_conf["user"])
pg_password = quote_plus(pg_conf["password"])
pg_host = pg_conf["host"]
pg_port = pg_conf["port"]
pg_db = pg_conf["db"]

conn_str = (
    f"postgresql+psycopg2://{pg_user}:{pg_password}@{pg_host}:{pg_port}/{pg_db}"
)
engine = create_engine(conn_str)
print("Connected to PostgreSQL:", pg_conf['host'], pg_conf['db'])

Connected to PostgreSQL: localhost storage_analytics


In [2]:
device_overview = pd.read_sql_query("SELECT * FROM mart_tableau_device_overview", engine)
anomaly_timeline = pd.read_sql_query("SELECT * FROM mart_tableau_anomaly_timeline", engine)
root_cause_summary = pd.read_sql_query("SELECT * FROM mart_tableau_root_cause_summary", engine)
grafana_health = pd.read_sql_query("SELECT * FROM v_grafana_device_health", engine)

In [3]:
device_overview.head()
anomaly_timeline.head()
root_cause_summary.head()
grafana_health.head()

,device,timestamp,total_iops,total_throughput_mb_s,avg_latency_ms,util_pct,aqu_sz,saturation_score,latency_pressure,anomaly_flag,critical_flag,high_flag
0,dm-0,2026-03-28 17:46:32.350044+00:00,218.715614,11.085472,0.729260,14.908920,0.798124,32.856686,1.311299,0,0,0
1,dm-0,2026-03-28 17:51:09.354490+00:00,215.675139,17.126012,0.884017,6.236243,1.447264,48.304067,2.163422,0,0,0
2,dm-0,2026-03-28 17:56:16.543551+00:00,193.383148,17.404994,0.645759,23.756002,0.273744,21.381845,0.822532,0,0,0
3,dm-0,2026-03-28 18:01:10.781559+00:00,189.305971,14.925893,0.820019,10.698208,0.297573,15.916329,1.064034,0,0,0
4,dm-0,2026-03-28 18:06:32.311664+00:00,115.579927,11.487333,1.234841,32.974840,0.282104,27.420221,1.583195,0,0,0


In [4]:
device_overview[["avg_latency_ms", "p95_latency_ms", "p99_latency_ms"]]
root_cause_summary.sort_values("anomaly_count", ascending=False).head(10)
grafana_health["anomaly_flag"].value_counts()

anomaly_flag
0    8962
1    1118
Name: count, dtype: int64